# Tone-Controlled Script Writer: Auto-generated Dataset + Unsloth Fine-tuning (Colab T4)

No external dataset is needed. The notebook:

1. Generates tone-labelled video scripts with the base Qwen2.5-7B-Instruct model (50 topics x 6 tones = 300 candidates, then filtered).
2. Reloads the model and fine-tunes it with QLoRA using Unsloth, so it learns to write in the tone requested in the prompt.
3. Lets you pick a tone and topic to generate a script.

**Before running:** Runtime > Change runtime type > **T4 GPU**. The `script_studio` folder in your Drive is created automatically.

In [ ]:
!pip install -q unsloth

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random
import torch

MODEL_NAME = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048

BASE_DIR = '/content/drive/MyDrive/script_studio'
DATA_PATH = os.path.join(BASE_DIR, 'train.jsonl')
OUTPUT_DIR = os.path.join(BASE_DIR, 'lora_adapter')
os.makedirs(BASE_DIR, exist_ok=True)

TONES = ['humorous', 'formal', 'inspirational', 'educational', 'dramatic', 'casual']

TONE_GUIDE = {
    'humorous': 'playful, witty, with light jokes and funny comparisons',
    'formal': 'professional and precise, no slang, no jokes',
    'inspirational': 'uplifting and motivating, speaks directly to the viewer about what they can achieve',
    'educational': 'clear and explanatory, teaches one or two concrete facts step by step',
    'dramatic': 'suspenseful and intense, short punchy sentences, builds tension',
    'casual': 'relaxed and conversational, like talking to a friend',
}

SYSTEM_PROMPT = 'You are a professional video script writer. Always write the script in the tone the user requests.'

print('GPU:', torch.cuda.get_device_name(0))

## 1. Topics

These 50 topics are used for every tone. Add or change topics to match your app's niche.

In [ ]:
TOPICS = [
    'why houseplants keep dying', 'benefits of drinking water in the morning',
    'how to start running as a beginner', 'tips for saving money on groceries',
    'how to prepare for a job interview', 'why sleep matters for productivity',
    'how to learn a new language in 15 minutes a day', 'the basics of budgeting',
    'how to make a healthy breakfast in five minutes', 'tips for working remotely',
    'how to write a good resume', 'why you should walk after meals',
    'how to reduce screen time', 'the history of the bicycle',
    'how to start a vegetable garden on a balcony', 'how to stay focused while studying',
    'why recycling plastic matters', 'how to make friends in a new city',
    'tips for first-time home buyers', 'how to care for a new puppy',
    'the basics of photography for beginners', 'how to prepare for a long flight',
    'why reading fiction improves empathy', 'how to speak confidently in public',
    'how to back up your phone photos', 'tips for cooking rice perfectly',
    'how to deal with procrastination', 'the science of why we yawn',
    'how to choose a good laptop for students', 'tips for better posture at a desk',
    'how solar panels generate electricity', 'how to start journaling',
    'why drinking tea can be relaxing', 'how to prepare for a marathon',
    'how to save energy at home', 'how to plan a weekly meal schedule',
    'the importance of hand washing', 'how to organize a small closet',
    'how to write a cold email that gets replies', 'why cities need more trees',
    'how to pick a good podcast to listen to', 'tips for traveling on a budget',
    'how to improve your handwriting', 'how bees pollinate flowers',
    'how to set goals you will actually achieve', 'tips for keeping a clean kitchen',
    'how to protect your passwords online', 'the benefits of learning to cook',
    'how to prepare for a school exam', 'why people love street food',
    'how to start investing with a small amount', 'tips for better sleep at night',
]

print(len(TOPICS), 'topics x', len(TONES), 'tones =', len(TOPICS) * len(TONES), 'candidates')

## 2. Load the base model and generate the tone-labelled dataset

Each script is generated with a system prompt that names the tone, then labelled with that tone. Scripts that are too short or too long are dropped. Generation is batched to save time.

In [ ]:
from unsloth import FastLanguageModel

base_model, base_tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)
base_tok.padding_side = 'left'
if base_tok.pad_token is None:
    base_tok.pad_token = base_tok.eos_token

def gen_messages(topic, tone):
    return [
        {'role': 'system', 'content': f'You are a professional video script writer. Write the script in a {tone} tone ({TONE_GUIDE[tone]}).'},
        {'role': 'user', 'content': f'Topic: {topic}\nWrite a short video script of about 120 words with a hook, two or three scenes, and a call to action. Output only the script text, with no title and no labels.'},
    ]

jobs = [(topic, tone) for topic in TOPICS for tone in TONES]
random.seed(3407)

rows = []
BATCH = 8
for i in range(0, len(jobs), BATCH):
    batch = jobs[i:i + BATCH]
    prompts = [
        base_tok.apply_chat_template(gen_messages(topic, tone), tokenize=False, add_generation_prompt=True)
        for topic, tone in batch
    ]
    enc = base_tok(prompts, return_tensors='pt', padding=True).to('cuda')
    with torch.no_grad():
        out = base_model.generate(
            **enc,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=base_tok.pad_token_id,
        )
    for (topic, tone), ids in zip(batch, out):
        text = base_tok.decode(ids[enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        if 60 <= len(text.split()) <= 300:
            rows.append({'tone': tone, 'topic': topic, 'script': text})
    print(f'{min(i + BATCH, len(jobs))}/{len(jobs)} generated, {len(rows)} kept')

random.shuffle(rows)
with open(DATA_PATH, 'w', encoding='utf-8') as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print('Saved', len(rows), 'rows to', DATA_PATH)
for tone in TONES:
    print(tone, sum(r['tone'] == tone for r in rows))

del base_model, base_tok
torch.cuda.empty_cache()

### Check a few samples

Read these before training. If a tone doesn't match its label, you can delete those rows from `train.jsonl` in Drive and re-run section 3 onward.

In [ ]:
with open(DATA_PATH, encoding='utf-8') as f:
    sample = [json.loads(line) for line in f][:3]
for r in sample:
    print(f"--- {r['tone']} | {r['topic']} ---")
    print(r['script'])
    print()

## 3. Reload the model for training (4-bit, Unsloth)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
model.print_trainable_parameters()

## 4. Format the data as chat conversations

The tone is written into the user message as `Tone: ...`. Inference uses the same format, which is how the model learns to follow the requested tone.

In [ ]:
from datasets import load_dataset

def build_user_message(tone, topic):
    return f'Tone: {tone}\nTopic: {topic}\nWrite a short video script.'

def to_text(example):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': build_user_message(example['tone'], example['topic'])},
        {'role': 'assistant', 'content': example['script']},
    ]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = load_dataset('json', data_files=DATA_PATH, split='train')
dataset = dataset.map(to_text, remove_columns=dataset.column_names)
print(len(dataset), 'training examples')

## 5. Train

The T4 doesn't support bf16, so training uses fp16. About 300 examples for 3 epochs takes roughly 20 to 40 minutes on a T4.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        bf16=False,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        logging_steps=5,
        seed=3407,
        output_dir='outputs',
        report_to='none',
    ),
)

stats = trainer.train()
print(stats)

## 6. Save the LoRA adapter to Google Drive

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved to', OUTPUT_DIR)

## 7. Generate scripts with a selected tone

Pick a tone from the dropdown and type a topic.

In [ ]:
FastLanguageModel.for_inference(model)

def generate_script(topic, tone, max_new_tokens=400):
    if tone not in TONES:
        raise ValueError(f'Tone must be one of {TONES}')
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': build_user_message(tone, topic)},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=True,
    ).to('cuda')
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        use_cache=True,
    )
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

from ipywidgets import interact

@interact(tone=TONES, topic='how to start running in the morning')
def run(tone, topic):
    print(f'--- Tone: {tone} ---\n')
    print(generate_script(topic, tone))

## 8. Optional: compare the same topic across tones

In [ ]:
topic = 'benefits of drinking water in the morning'
for tone in TONES[:3]:
    print(f'=== {tone.upper()} ===')
    print(generate_script(topic, tone, max_new_tokens=250))
    print()

## 9. Optional: reload the saved adapter in a new session

Run this instead of sections 2 to 5 so you don't have to retrain.

In [ ]:
# from unsloth import FastLanguageModel
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=OUTPUT_DIR,
#     max_seq_length=MAX_SEQ_LENGTH,
#     load_in_4bit=True,
# )
# FastLanguageModel.for_inference(model)